# Exercise 5 — Formulate, calibrate, estimate, control and debug a CSTR

Seven linked tasks develop a complete temperature-feedback workflow. The physical
model below is visible and editable; optimization formulations are your work.
Complete the tasks in order, preserve units and time indexing, and explain what
each result says about the model/data/solver before changing an option.


### Prepare an independent notebook

Run cells from top to bottom in a fresh kernel. This notebook defines its own
physical equations, estimation problem and numerical experiments; it does not
import a model or execute another notebook. Repository helpers only display
figures, create optional controls and summarize solver statistics.

NumPy arrays below store time along rows, while CasADi decision matrices store
states along rows and time along columns. `DM.full()` crosses the boundary from
CasADi numerical matrices to NumPy. Plotly provides hover, zoom and trace toggles;
`CASADI_TUTORIAL_PLOTS=static` selects Matplotlib and
`CASADI_TUTORIAL_WIDGETS=0` hides optional controls. All default results are
computed by ordinary functions before any Run button is created.


In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter
from functools import partial

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np

repository_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from casadi_tutorial.diagnostics import max_bound_violation, solver_summary
from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_series

# Every trajectory time axis in the CSTR lessons is measured in hours.
plot_series = partial(plot_series, xlabel="time [h]")

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)


### Derive the well-mixed reactor balances and fix the units

Assume constant liquid volume and density, equal inlet/outlet flow, perfect
mixing, one irreversible first-order reaction, and an imposed jacket temperature.
The state is $x=[C_A,T]^\mathsf T\in\mathbb R^2$, input $u=T_c\in\mathbb R$,
known disturbance $d=[C_{Af},T_f]^\mathsf T\in\mathbb R^2$, and the uncertain
parameter is total heat-transfer conductance $U_A$.

| Quantity | Value | Units |
|---|---:|---|
| Flow $F$, volume $V$ | 1, 1 | m³/h, m³ |
| Arrhenius prefactor $k_0$ | $9703\times3600$ | h⁻¹ |
| Activation energy $E$, gas constant $R$ | 11843, 1.98589 | kcal/kmol, kcal/(kmol K) |
| Reaction enthalpy $\Delta H$ | −5960 | kcal/kmol |
| Volumetric heat capacity $\rho C_p$ | 500 | kcal/(m³ K) |
| True $U_A$ | 150 | kcal/(h K) |
| Feed $C_{Af},T_f$ | 10, 298.15 | kmol/m³, K |

The rate and material/energy balances are

$$r=k_0e^{-E/(RT)}C_A,\qquad
\dot C_A=\frac FV(C_{Af}-C_A)-r,$$
$$\dot T=\frac FV(T_f-T)-\frac{\Delta H}{\rho C_p}r
+\frac{U_A}{V\rho C_p}(T_c-T).$$

Each material-balance term has units kmol/(m³ h); each energy-balance term has
units K/h. Because $\Delta H<0$, reaction releases heat. The jacket removes heat
when $T_c<T$. Concentration and absolute temperature must remain physical; a
numerically converged trajectory outside the stated domain is unsuitable.

**Predict:** increasing the jacket temperature initially changes which balance?
Which sign in the energy balance would accidentally turn an exothermic reaction
into cooling?


In [ ]:
FLOW = 1.0
VOLUME = 1.0
K0 = 9703.0 * 3600.0
ACTIVATION_ENERGY = 11843.0
GAS_CONSTANT = 1.98589
REACTION_ENTHALPY = -5960.0
HEAT_CAPACITY = 500.0
TRUE_UA = 150.0
X_NOMINAL = np.array([8.5695, 311.267])
U_NOMINAL = 298.15
D_NOMINAL = np.array([10.0, 298.15])
X_SCALE = np.array([0.5, 5.0])
U_SCALE = 10.0
STATE_LOWER = np.array([0.0, 285.0])
STATE_UPPER = np.array([12.0, 360.0])
INPUT_LOWER, INPUT_UPPER = 285.0, 315.0


def cstr_rhs(state, jacket_temperature, heat_transfer, disturbance):
    """Visible species/energy balances; time in h, temperature in K."""
    concentration, temperature = state[0], state[1]
    feed_concentration, feed_temperature = disturbance[0], disturbance[1]
    rate = K0 * ca.exp(-ACTIVATION_ENERGY / (GAS_CONSTANT * temperature)) * concentration
    material_balance = FLOW / VOLUME * (feed_concentration - concentration) - rate
    energy_balance = (
        FLOW / VOLUME * (feed_temperature - temperature)
        - REACTION_ENTHALPY / HEAT_CAPACITY * rate
        + heat_transfer / (VOLUME * HEAT_CAPACITY) * (jacket_temperature - temperature)
    )
    return ca.vertcat(material_balance, energy_balance)


### Turn the balances into differentiable continuous and discrete maps

Named CasADi inputs make the dimensions explicit: `x` is 2×1, `u` and `ua`
are scalars, `d` is 2×1, and `h` is a scalar measured in hours. Keeping `ua` as a
function input lets the same graph participate in parameter estimation.

A zero-order hold keeps $u,d$ fixed over each interval. Classical RK4 computes
$k_1=f(x)$, $k_2=f(x+hk_1/2)$, $k_3=f(x+hk_2/2)$,
$k_4=f(x+hk_3)$ and
$F_h=x+h(k_1+2k_2+2k_3+k_4)/6$. This approximation is differentiable but its
accuracy must be checked separately from solver convergence.

`jacobian` gives $A=\partial f/\partial x$, $B=\partial f/\partial u$ and
$E_d=\partial f/\partial d$. They describe perturbations about an operating
point, not absolute state evolution.


In [ ]:
x_symbol = ca.MX.sym("x", 2)
u_symbol = ca.MX.sym("u")
ua_symbol = ca.MX.sym("ua")
d_symbol = ca.MX.sym("d", 2)
h_symbol = ca.MX.sym("h")
xdot = cstr_rhs(x_symbol, u_symbol, ua_symbol, d_symbol)
f = ca.Function("cstr_balances", [x_symbol, u_symbol, ua_symbol, d_symbol], [xdot],
                ["x", "u", "ua", "d"], ["xdot"])
k1 = xdot
k2 = cstr_rhs(x_symbol + h_symbol * k1 / 2, u_symbol, ua_symbol, d_symbol)
k3 = cstr_rhs(x_symbol + h_symbol * k2 / 2, u_symbol, ua_symbol, d_symbol)
k4 = cstr_rhs(x_symbol + h_symbol * k3, u_symbol, ua_symbol, d_symbol)
F = ca.Function("cstr_rk4", [x_symbol, u_symbol, ua_symbol, d_symbol, h_symbol],
                [x_symbol + h_symbol * (k1 + 2*k2 + 2*k3 + k4) / 6],
                ["x", "u", "ua", "d", "h"], ["x_next"])
linearization = ca.Function("cstr_jacobians", [x_symbol, u_symbol, ua_symbol, d_symbol],
    [ca.jacobian(xdot, x_symbol), ca.jacobian(xdot, u_symbol), ca.jacobian(xdot, d_symbol)])


### Keep simulation, validation and presentation separate

`simulate_cstr` returns an $(N+1)\times2$ numerical history for $N$ held inputs.
The first row is the initial state, so measurements and state plots have one
more point than the input sequence. The held-input plot repeats its final value
at the last state time so the last interval remains visible.

`constraint_violation` independently evaluates every Opti constraint against its
bounds in the solver's scaled coordinates. A successful return status alone
does not check finite numbers or physical constraints. Objective normalization
is distinct from decision/constraint scaling; subsequent NLPs use dimensionless
state decisions $z=S_x^{-1}(x-x_s)$, $S_x=\operatorname{diag}(0.5,5)$.


In [ ]:
IPOPT_OPTIONS = {"print_level": 0, "sb": "yes", "max_iter": 200, "tol": 1e-7,
                 "bound_relax_factor": 0.0, "constr_viol_tol": 1e-8}


def simulate_cstr(initial_state, inputs, heat_transfer=TRUE_UA, step_size=0.1,
                  disturbance=D_NOMINAL):
    states = [np.asarray(initial_state, dtype=float).copy()]
    for jacket in np.asarray(inputs).reshape(-1):
        states.append(np.asarray(F(states[-1], jacket, heat_transfer, disturbance, step_size)).ravel())
    return np.asarray(states)


def constraint_violation(problem, solution):
    return max_bound_violation(
        np.asarray(solution.value(problem.g)).ravel(),
        np.asarray(solution.value(problem.lbg)).ravel(),
        np.asarray(solution.value(problem.ubg)).ravel(),
    )


def show_trajectory(states, inputs, step_size, title, measured_temperature=None):
    time = step_size * np.arange(len(states))
    display_plot(plot_series(time, {"concentration": states[:, 0]},
        title=title, ylabel="C_A [kmol/m³]"))
    temperatures = {"temperature": states[:, 1]}
    if measured_temperature is not None:
        temperatures["noisy observation"] = measured_temperature
    display_plot(plot_series(time, temperatures, title=title, ylabel="T [K]"))
    if len(inputs):
        held = np.r_[inputs, inputs[-1]]
        display_plot(plot_series(time, {"jacket": held},
            title=title, ylabel="T_c [K]", steps=True))


### Task 1 — Re-derive and validate the model

Starting from total material and energy accumulation, explain every term in
`cstr_rhs`, including units and the exothermic sign. Evaluate the nominal residual
and form $A,B,E_d$ using CasADi Jacobians. Check a residual below $2\times10^{-4}$
in the documented units and discuss the slow local time scale. Display both
concentration and temperature plus the held jacket input after a +5 K step.


In [ ]:
# TODO: evaluate f, construct/evaluate Jacobians, simulate the held +5 K input,
# and plot state/input histories with units. Explain the signs before coding.


### Task 2 — Test linearization and integration accuracy

Use $\delta\dot x=A\delta x+B\delta u$ to compare +5 K and +15 K steps against
the nonlinear balances. Then compare RK4 meshes $h=0.1,0.05,0.025$ h on coincident
sample times. State which error is local-model error and which is integration
error; do not attribute either automatically to IPOPT.


In [ ]:
# TODO: implement perturbation simulation and mesh refinement.
# Expected shapes: states (N+1, 2), held inputs (N,).


### Task 3 — Fit heat transfer and check information

Generate 6 h of data using 15-sample plateaus at jacket temperatures
298.15, 303.15, 293.15 and 301.15 K, $h=0.1$ h, seed 7 and 0.05 K temperature
noise. Fit $50\le U_A\le300$ using scaled state decisions and dynamic defects.
Initialize by simulating $U_A=100$; do not use the true state trajectory as an
initial guess. Plot the objective profile, residuals and sensitivity; validate
on a separate three-plateau record. Explain why constant nominal input can still
inform this single-parameter fit when other physics and initial state are known.


In [ ]:
# TODO: formulate the multiple-shooting least-squares problem visibly.
# Targets: relative U_A error <5%; held-out temperature RMSE <0.1 K.


### Task 4 — Reconstruct the unmeasured state with MHE

Formulate the temperature-only MHE in chapter 14a using a ten-interval growing
window. Begin with prior $[8,309]$, while truth starts at $x_s+[0.5,-2]$. Use
arrival scales $[1,0.5]$, interval model-error scales $[0.002,0.015]$ in state
units and sensor noise 0.05 K. Explain why the shifted fixed arrival cost is an
approximation. Keep true states outside the estimator API.

Check local observability rank two; report RMSE after 1 h and all solver statuses.
Inject one missing result and then two consecutive missing results; explain the
propagation and stop policy.


In [ ]:
# TODO: define the MHE variables, measurement map, arrival cost, dynamics and update.
# Targets: post-startup RMSE <[0.08 kmol/m³, 0.06 K], finite feasible estimates.


### Task 5 — Compare estimated-state NMPC with ideal feedback

Use the fitted parameter in both controllers. Formulate scaled multiple shooting
with $h=0.1$ h, horizon 20, state bounds $[0,285]\le x\le[12,360]$, jacket bounds
285–315 K and objective $\sum\|z_j\|^2+0.1\sum v_j^2$. Run 60 moves, applying
only the first move and shifting the previous prediction. Give one branch exact
state and the other only MHE state. Plot actual/estimated/predicted states,
applied inputs, iteration counts and primal residuals.

Compare integrated normalized tracking error; the deterministic estimated-state
result should be within 1.2 times ideal feedback, with all bounds satisfied.


In [ ]:
# TODO: build the controller locally and implement the measure-estimate-solve-apply loop.
# Truth belongs to the simulator/evaluation; do not pass it into MHE.


### Task 6 — Add an honest soft temperature preference

Add $s_j\ge0$, $T_j\le308+s_j$ and $100\sum(s_j/1\mathrm K)^2$ to the OCP while
preserving the 360 K hard model-domain bound. Include node zero so an initially
conflicting preference is represented explicitly. Plot temperature and slack;
verify the softened inequality numerically and explain why a large penalty
cannot change the initial state or create cooling authority.


In [ ]:
# TODO: solve the soft-limit OCP and report maximum slack [K] and primal residual.


### Task 7 — Diagnose a genuine failure before changing tolerances

Force an IPOPT iteration-limit failure with `max_iter=0` and inspect
`return_status`. Reject the returned candidate solution, evaluate a bounded
one-step fallback and stop after a second consecutive failed control solve.
Separately diagnose $T_0=370$ K as a node-zero hard-bound contradiction.
Explain which remedies fit each cause and why they differ.

Compare your work with the [complete solution](../solutions/05_cstr.ipynb) only
after completing each task. Every requested estimator/controller/soft-constraint
and failure experiment is worked there.


In [ ]:
# TODO: trigger/catch failures deliberately, record diagnostics and assert the stop policy.
